In [ ]:
!pip install addfips
!pip install us

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 10.7 MB/s eta 0:00:00


In [ ]:
# standard stuff
import numpy as np
import pandas as pd
# country Map
import plotly.express as px
import addfips
import us
# Regretion stuff
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
#gradient boosting stuff
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
VegData = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vTU6WAz3UfcApC4kZVCDhdYwU8FemgPqRv4DKM9QqAwoeD3xZbCSBRxCqgYlunKUw/pub?gid=787104991&single=true&output=csv')
df = pd.DataFrame(VegData)
df = df.dropna(subset=['subnational1', 'subnational2'])
df['Fips Code'] = np.nan
print(df.columns)

Index(['country', 'subnational1', 'subnational2', 'threshold', 'area_ha',
       'extent_2000_ha', 'extent_2010_ha', 'gain_2000-2020_ha',
       'tc_loss_ha_2001', 'tc_loss_ha_2002', 'tc_loss_ha_2003',
       'tc_loss_ha_2004', 'tc_loss_ha_2005', 'tc_loss_ha_2006',
       'tc_loss_ha_2007', 'tc_loss_ha_2008', 'tc_loss_ha_2009',
       'tc_loss_ha_2010', 'tc_loss_ha_2011', 'tc_loss_ha_2012',
       'tc_loss_ha_2013', 'tc_loss_ha_2014', 'tc_loss_ha_2015',
       'tc_loss_ha_2016', 'tc_loss_ha_2017', 'tc_loss_ha_2018',
       'tc_loss_ha_2019', 'tc_loss_ha_2020', 'tc_loss_ha_2021',
       'tc_loss_ha_2022', 'tc_loss_ha_2023', 'Fips Code'],
      dtype='object')


In [ ]:
af = addfips.AddFIPS() # Initialize AddFIPS once for efficiency

# Iterate over the index and row data using iterrows()
for index, row_data in VegData.iterrows():
    State = row_data['subnational1']
    County = row_data['subnational2']
    # Get Fips code, handling cases where state/county might not be found by addfips
    Fips = af.get_county_fips(County, state=State)
    VegData.loc[index, 'Fips Code'] = Fips

In [ ]:
VegParams = df.loc[:,'tc_loss_ha_2001':'tc_loss_ha_2016'].columns


for param in VegParams:
    county_avg = VegData.groupby('subnational2')[param].mean().reset_index()#this
    county_avg['Fips Code'] = county_avg['Fips Code'].astype(str).str.zfill(5)

    fig = px.choropleth(
        county_avg,
        geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
        locations='Fips Code',
        featureidkey = 'id',
        color= param,
        scope='usa',
        title=f'{param} by County (2000–2016)',
        color_continuous_scale='RdYlBu_r',
        range_color=(0, 10000),
        labels={'Arithmetic Mean': param}
    )
    fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()

In [ ]:
for param in VegParams:
    State_avg = VegData.groupby('subnational1')[param].mean().reset_index()

    # Function to get state abbreviation, handling 'District of Columbia' and None cases
    def get_state_abbr_or_original(state_name):
        if state_name == 'District of Columbia':
            return 'DC'
        state_obj = us.states.lookup(state_name)
        if state_obj:
            return state_obj.abbr
        return state_name # Fallback to original name if not found

    State_avg['subnational1'] = State_avg['subnational1'].apply(get_state_abbr_or_original)

    fig = px.choropleth(
        State_avg,
        locations='subnational1',
        locationmode='USA-states',
        color= param,
        scope='usa',
        title=f'{param} by State (2000–2016)',
        color_continuous_scale='RdYlBu_r',
        range_color=(0, 10000),
        labels={'Arithmetic Mean': param}
    )
    fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()

#Lasoo Regression

In [ ]:
X = 4 #imput columns wanted to be calced
Target = 4 #imput copd deaths by county

In [ ]:
# can add more conditions, to keep counties aligned must pass counties as its own df and request counties_trian and Counties_test

X_train, X_test, y_train, y_test = train_test_split(
    X, Target, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model = Lasso(alpha=1.0)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print(y_pred)
print(y_test)

# Gradient Boosting
uses some of the lasso regrestion code blocks as a foundation

In [ ]:
model = GradientBoostingRegressor()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [ ]:
print(y_pred)
print(y_test)